# Population map in GeoLibre

Shows the population raster on an interactive web map using
[GeoLibre](https://github.com/opengeos/geolibre).

The raster (`data/processed/population_web_z13.tif`, built by
`scripts/08_web_raster.py`) is in Web Mercator (EPSG:3857) on the web map's own
zoom-13 tile grid: cells of about 10-12 m on the ground across GB. Because every
cell lines up with a map tile, GeoLibre draws it without resampling.

Everything runs locally: GeoLibre starts a small web server on your machine,
which serves the GeoLibre app and reads the GeoTIFF straight from disk.

Requirements: `pip install geolibre`, run in Jupyter or VS Code. To open the same
map in your web browser instead, run `python scripts/09_geolibre_map.py`.

In [ ]:
from pathlib import Path

import geolibre
import geolibre._server
from geolibre._server import register_local_file
from IPython.display import IFrame

PROCESSED = Path.cwd().parent / "data" / "processed"
RASTER = PROCESSED / "population_web_z13.tif"
PAGE = PROCESSED / "geolibre_map_notebook.html"
# Python's default backlog (5) makes GeoLibre's local server refuse bursts of
# tile requests, leaving holes in the map; set before the first Map
geolibre._server._QuietServer.request_queue_size = 128

assert RASTER.exists(), f"{RASTER} not found: run scripts/08_web_raster.py first"

## The map

Half of the populated cells hold 3 people or fewer and 99% hold 16 or fewer, so
the colour ramp runs from 1 to 20 people per cell; busier cells (blocks of flats,
halls of residence) all show in the darkest red. Empty cells are nodata and
stay transparent over the fiord basemap. Zoomed out, each pixel shows the average
of the populated cells beneath it.

The map is shown as a plain web page in a frame rather than as a notebook
widget: VS Code downloads widget code from a CDN, which can fail, while the
frame only needs GeoLibre's local server.

GeoLibre's **Identify** tool (pointer icon above the layer list) shows the
Population of any cell you click.

Running the next cell shows the live map. On GitHub, where it can't run, this is what it
looks like (central London):

![Population in central London](../docs/images/map_london.png)

In [ ]:
m = geolibre.Map(center=[-2.5, 54.5], zoom=6, basemap="fiord", layout="full")
m.add_cog(RASTER, "Population", colormap="reds", rescale=[[1, 20]])

# Lock the view to GB: no panning outside this lon/lat box or zooming out past it
prefs = m.project["preferences"]
prefs["map"].update(restrictBounds=True, bounds=[-9.0, 49.5, 2.5, 61.0], minZoom=4.5)
m.project = {**m.project, "preferences": prefs}

# Write the map as a page, serve it from GeoLibre's local server and show it.
# The page works only while this notebook's kernel is running.
m.to_html(str(PAGE), title="Population map", height="100vh", app_url=m._app_url)
IFrame(register_local_file(PAGE), width="100%", height=750)